# Cryptography (CC4017) -- Week 5

## Q1: Collision resistant Hash Functions

Consider H : M → T a collision resistant hash function that takes messages of any size m ∈ M = {0, 1}*
and produces outputs with 64 bit length t ∈ T = {0, 1}⁶⁴.
1. H′ = (H(m) || H(m) || H(m))
2. H′ = H(m || m || m)
3. H′ = H(64)
4. H′ = H(m||64)
5. H′ = H(m)[0 . . . 10] // truncate the output to 10 bits
6. H′ = H(m[0 . . . |m|-2]) // hash m without its last bit
7. H′ = H(m) || H(m ⊕ 1|m| )
8. H′ = H(m) if m = 064 ∧ m = 164 , H(m ⊕ 1|m| ) otherwise

Question: Which of the proposed hash constructions H′ are also collision resistant?

1. Is collision resistant, since it concatenates the result of hashing m with H. Since the result of H(m) is collision resistant so is the concatenation of 3 collision resistant hashes
2. Is collision resistant, since H is collision resistant, and the 2nd hash works as the implementation of H over the message concatenated with itself twice. This means that for it to not be collision resistant, H isn't collision resistant, and its already been stablished before that it is.
3. Isn't collision resistant, H' will, indepently from the value of m, produce H(64), meaning all messages will give the same hash and, therefore, collide.
4. Is collision resistant, thanks to the fact that its still hashing a unique message followed by 64. H is collision resistant therefore so is this architecture.
5. Isn't collision resistant, because by truncating the output to onoly 10 bits, we reducing the realms of possible hashes from 2⁶⁴ to 2¹⁰ making it many times more prone for collisions to happen.
6. Ins't collision resistant, since as long as two messages are produced only differing the last bit, they will collide, because said bit is removed before hashing with H.
7. Is collision resistant, due to the fact that no m can be equal to its negation, swapping its bites for the oppsite, the same as XOR'ing with 1^m. In this its impossible to produce a message that when its hash through H is concatenated with its opposited hashed with H, will be the same as another message, since H is collision resistant.
8. Is collision resistant, following the same logic as before, all hashes producted by H over the negation of m will be unique. The edge cases 0⁶⁴ and 1⁶⁴ are the opposite of each other, meaning there is no case where u can produce a message in which its negation will produce the same hash.

## Q2: Rho method to find Hash collisions

As described in [1], the Rho method is an algorithm for finding collisions that, unlike the naive birthday
attack, requires only a small amount of memory. To find collision in hash function H(m), it works as
follows.
1. Given a hash function with n-bit values, pick some random hash value h1 and define h′1 = h1 .
2. Compute h2 = H(h1 ) and h′2 = H(H(h′1 )). In the first case, we apply the hash function once. In
the second, we apply it twice.
3. Iterate the process and compute hi+1 = H(hi ) and h′i+1 = H(H(h′i )), until you reach a i such
that h′i+1 = hi+1
4. If this is the case, then you have found a loop within the possible hash values. How can we find
the collision now? Check out this proof.


Complete the code in rho_exercise.py to do this.

- You must complete function rho, which is parametrized by an initial value
- Function H computes hashes truncated as necessary.
- You can adjust the global parameter during testing, but the goal is to find a collision in L = 5.

Also include a succinct analysis of how long it takes to find these collisions, both in cycle iterations
and real time. How does this scale with L?


In [21]:
from cryptography.hazmat.primitives import hashes
import os

L = 5 # output length in bytes

# Something to make calling hash functions more succint
def H(X):
	digest = hashes.Hash(hashes.SHA256())
	digest.update(X)
	return (digest.finalize()[0:L])

# Write a function that finds the collision and presents the values in which it occurred
def rho(h0):
    print("Hash is "+str(8*L)+" bits")
    # Your code here!!
    h0 = H(h0)
    hi = H(H(h0))
    while True:
        if(hi == h0):
            break
        else:
            h0 = H(h0)
            hi = H(H(hi))
            
    return (h0, hi)

start = os.urandom(L)
(h0, h1) = rho(start)
print(h0)
print(h1)

Hash is 40 bits
b'\x05~\xcfP\x80'
b'\x05~\xcfP\x80'


For a hash output truncated to L bytes, the output space is 2⁸^L, meaning its expected to be around 2⁴^L iterations in order to find a collision. Having in account that an iteration takes about 0.1 miliseconds so the time it takes for the program to run is expected to be 2⁴^L * 0.1 ms. By doing the math we see it should take the program around 1.7 minutes, for L=5, which is the times we are getting when running the code.

## Q3: Weak ciphers

The code in ciphersuite_fsr.py contains a very poorly implemented “stream cipher’ ’.

In [22]:
import os
import random
from cryptography.hazmat.primitives import hashes

# Internal state of the LFSR
x = 1

# Call the lfsr to update the internal state
def __lfsr():
	global x
	x = (x**5 + x**4 + 1) % 1009

# Use crypto random generation to initialize the LFSR
def gen(): 
	global x
	sysrand = random.SystemRandom()
	x = sysrand.randint(0,1008)

# Bitwise XOR operation.
def enc(m):
	global x
	k = b""
	while (len(k) < len(m)):
		__lfsr()
		digest = hashes.Hash(hashes.SHA256())
		digest.update(x.to_bytes(4, "big"))
		k += digest.finalize()
	return bytearray([a ^ b for a,b in zip(m,k[:len(m)])])

# Reverse operation
def dec(c):
	global x
	k = b""
	while (len(k) < len(c)):
		__lfsr()
		digest = hashes.Hash(hashes.SHA256())
		digest.update(x.to_bytes(4, "big"))
		k += digest.finalize()
	return bytearray([a ^ b for a,b in zip(c, k[:len(m)])])


### Question - P1: 
Consider the IND-CPA security experiment. How many calls to the encryption oracle do you have to do to succeed?

The number of calls made to the oracle will depend on the period of the lsfr. As it is known, Linear Feedback Shift Registers are periodical, that is, their internal state follows a cycle and the values it produces will eventually repeat in the same order. As long as the attacker knows this, all they have to do is send p + 2 calls to the oracle. With p being the period, the attacker does p + 1 calls to iterate over each possible state and to reach the initial one, confirming from now on the states will repeat. As they reach that, they will know for sure, what the next state will be, allowing them to simply make another last call, the attack, where they will, for sure, guess what message is gonna be encrypted.

### Question - P2: 
Describe how one can construct an attacker against the IND-CPA experiment running this encryption scheme.

Due to the periodical nature of LSFR, an attacker could be 100% sure of what message is being encrypted. With a prepared message of M0, the same size as the internal state of the LSFR, which can be determined or guess by techniques like statistical analysis, the attacker simply has to send M0 to the oracle the p + 1 times mentioned before, where p is the period of the LSFR. Since the attacker now knows what would result in encrypting M0 on the next LSFR internal state, due to going through every state of the system and using it to encrypt M0, all the attacker has to do left is send M0 and M1 a message that simply has to be different than M0. When looking at Cb, if the cyphertext is the value that was expected, that it corresponds to C0, and b = 0, otherwise b = 1.